# SH17 PP-PicoDet-L Portable Benchmark

This notebook benchmarks PP-PicoDet-L on SH17 as a non-YOLO, non-EfficientDet lightweight detector. PP-PicoDet-L has about 5.80M parameters, below the 7.2M parameter budget of YOLOv9s, and official PaddleDetection configs support 320, 416, and 640 input sizes.

The experiment sequence mirrors the existing SH17 ablation style:

1. `picodet_l_320_baseline`
2. `picodet_l_416_scale`
3. `picodet_l_640_scale`
4. `picodet_l_640_oversample_minority`

Full training is disabled by default. Set `RUN_TRAINING = True` after checking paths and GPU setup.

In [ ]:
from pathlib import Path
import os

# Portable paths. Override with environment variables or edit these values in the notebook.
PROJECT_ROOT = Path.cwd()
DATA_ROOT = os.environ.get("SH17_DATA_ROOT", PROJECT_ROOT)
DATA_ROOT = Path(DATA_ROOT).expanduser()
TRAIN_MANIFEST = Path(os.environ.get("SH17_TRAIN_MANIFEST", DATA_ROOT / "train_files.txt")).expanduser()
VAL_MANIFEST = Path(os.environ.get("SH17_VAL_MANIFEST", DATA_ROOT / "val_files.txt")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("SH17_OUTPUT_ROOT", PROJECT_ROOT / "outputs" / "picodet_l_paddledet")).expanduser()
PADDLEDET_ROOT = Path(os.environ.get("PADDLEDET_ROOT", OUTPUT_ROOT / "PaddleDetection")).expanduser()

AUTO_SETUP = True
USE_GPU = True
SEED = 0
EPOCHS = 200
RUN_TRAINING = False
RUN_EXPERIMENT_NAMES = [
    "picodet_l_320_baseline",
    "picodet_l_416_scale",
    "picodet_l_640_scale",
    "picodet_l_640_oversample_minority",
]

EXPECTED_CLASS_NAMES = [
    "person",
    "ear",
    "ear-mufs",
    "face",
    "face-guard",
    "face-mask",
    "foot",
    "tool",
    "glasses",
    "gloves",
    "helmet",
    "hands",
    "head",
    "medical-suit",
    "shoes",
    "safety-suit",
    "safety-vest",
]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("PADDLEDET_ROOT:", PADDLEDET_ROOT)

In [ ]:
import sys
from pathlib import Path

HELPER_MODULE_RELATIVE_PATH = Path("scripts") / "sh17_picodet_dataset.py"
SELF_CONTAINED_HELPER_SOURCE = 'from __future__ import annotations\n\nimport copy\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Callable, Iterable\n\n\nCLASS_NAMES = [\n    "person",\n    "ear",\n    "ear-mufs",\n    "face",\n    "face-guard",\n    "face-mask",\n    "foot",\n    "tool",\n    "glasses",\n    "gloves",\n    "helmet",\n    "hands",\n    "head",\n    "medical-suit",\n    "shoes",\n    "safety-suit",\n    "safety-vest",\n]\n\nMINORITY_CLASS_IDS_ZERO_BASED = {2, 4, 6, 10, 13, 16}\nMINORITY_REPEAT_FACTOR = 3\nPICODET_L_PARAMS_M = 5.80\n\n\n@dataclass(frozen=True)\nclass PicodetExperiment:\n    name: str\n    input_size: int\n    train_json_name: str\n    batch_size: int\n    base_lr: float\n    purpose: str\n    config_file_name: str\n\n    @property\n    def official_config_name(self) -> str:\n        return f"picodet_l_{self.input_size}_coco_lcnet.yml"\n\n\ndef picodet_experiments() -> list[PicodetExperiment]:\n    return [\n        PicodetExperiment(\n            name="picodet_l_320_baseline",\n            input_size=320,\n            train_json_name="instances_train.json",\n            batch_size=24,\n            base_lr=0.12,\n            purpose="baseline lightweight PP-PicoDet-L at 320 input",\n            config_file_name="picodet_l_320_sh17.yml",\n        ),\n        PicodetExperiment(\n            name="picodet_l_416_scale",\n            input_size=416,\n            train_json_name="instances_train.json",\n            batch_size=24,\n            base_lr=0.12,\n            purpose="moderate-resolution scale variant for small PPE objects",\n            config_file_name="picodet_l_416_sh17.yml",\n        ),\n        PicodetExperiment(\n            name="picodet_l_640_scale",\n            input_size=640,\n            train_json_name="instances_train.json",\n            batch_size=12,\n            base_lr=0.06,\n            purpose="high-resolution PP-PicoDet-L scale variant",\n            config_file_name="picodet_l_640_sh17.yml",\n        ),\n        PicodetExperiment(\n            name="picodet_l_640_oversample_minority",\n            input_size=640,\n            train_json_name="instances_train_oversampled.json",\n            batch_size=12,\n            base_lr=0.06,\n            purpose="high-resolution variant with minority-class oversampling",\n            config_file_name="picodet_l_640_oversample_sh17.yml",\n        ),\n    ]\n\n\ndef _path_for_yaml(path: Path) -> str:\n    return str(path).replace("\\\\", "/")\n\n\ndef read_manifest(manifest_path: Path, dataset_root: Path) -> list[Path]:\n    image_paths: list[Path] = []\n    for raw in manifest_path.read_text(encoding="utf-8").splitlines():\n        item = raw.strip()\n        if not item:\n            continue\n        candidate = Path(item)\n        if not candidate.is_absolute():\n            candidate = dataset_root / item\n        if not candidate.exists():\n            fallback = dataset_root / "images" / candidate.name\n            if fallback.exists():\n                candidate = fallback\n        image_paths.append(candidate)\n    return image_paths\n\n\ndef default_image_size_resolver(image_path: Path) -> tuple[int, int]:\n    from PIL import Image\n\n    with Image.open(image_path) as image:\n        return image.size\n\n\ndef label_path_for_image(image_path: Path, dataset_root: Path) -> Path:\n    candidates = [\n        dataset_root / "labels" / f"{image_path.stem}.txt",\n        image_path.with_suffix(".txt"),\n        image_path.parent / "labels" / f"{image_path.stem}.txt",\n        image_path.parent.parent / "labels" / f"{image_path.stem}.txt",\n    ]\n    for candidate in candidates:\n        if candidate.exists():\n            return candidate\n    return candidates[0]\n\n\ndef _relative_file_name(image_path: Path, dataset_root: Path) -> str:\n    try:\n        return image_path.relative_to(dataset_root).as_posix()\n    except ValueError:\n        return image_path.name\n\n\ndef _validate_yolo_values(\n    class_id: int,\n    xc: float,\n    yc: float,\n    bw: float,\n    bh: float,\n    label_path: Path,\n) -> None:\n    if class_id < 0 or class_id >= len(CLASS_NAMES):\n        raise ValueError(f"{label_path}: class id {class_id} outside [0, {len(CLASS_NAMES) - 1}]")\n    values = {"xc": xc, "yc": yc, "bw": bw, "bh": bh}\n    for name, value in values.items():\n        if value < 0.0 or value > 1.0:\n            raise ValueError(f"{label_path}: {name}={value} outside [0, 1]")\n    if bw <= 0.0 or bh <= 0.0:\n        raise ValueError(f"{label_path}: bbox width/height must be positive")\n\n\ndef yolo_label_rows(label_path: Path) -> list[tuple[int, float, float, float, float]]:\n    if not label_path.exists():\n        raise FileNotFoundError(f"Missing label file: {label_path}")\n\n    rows: list[tuple[int, float, float, float, float]] = []\n    for line_number, raw in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):\n        line = raw.strip()\n        if not line:\n            continue\n        parts = line.split()\n        if len(parts) < 5:\n            raise ValueError(f"{label_path}:{line_number}: expected 5 YOLO label values")\n        class_id = int(float(parts[0]))\n        xc, yc, bw, bh = [float(value) for value in parts[1:5]]\n        _validate_yolo_values(class_id, xc, yc, bw, bh, label_path)\n        rows.append((class_id, xc, yc, bw, bh))\n    return rows\n\n\ndef convert_manifest_to_coco(\n    manifest_path: Path,\n    dataset_root: Path,\n    output_json: Path,\n    image_size_resolver: Callable[[Path], tuple[int, int]] | None = None,\n) -> dict[str, int]:\n    resolver = image_size_resolver or default_image_size_resolver\n    image_paths = read_manifest(manifest_path, dataset_root)\n\n    images = []\n    annotations = []\n    annotation_id = 1\n\n    for image_id, image_path in enumerate(image_paths, start=1):\n        width, height = resolver(image_path)\n        images.append(\n            {\n                "id": image_id,\n                "file_name": _relative_file_name(image_path, dataset_root),\n                "width": int(width),\n                "height": int(height),\n            }\n        )\n\n        label_path = label_path_for_image(image_path, dataset_root)\n        for class_id, xc, yc, bw, bh in yolo_label_rows(label_path):\n            box_w = bw * width\n            box_h = bh * height\n            x_min = (xc - bw / 2.0) * width\n            y_min = (yc - bh / 2.0) * height\n            annotations.append(\n                {\n                    "id": annotation_id,\n                    "image_id": image_id,\n                    "category_id": class_id + 1,\n                    "bbox": [\n                        round(x_min, 6),\n                        round(y_min, 6),\n                        round(box_w, 6),\n                        round(box_h, 6),\n                    ],\n                    "area": round(box_w * box_h, 6),\n                    "iscrowd": 0,\n                }\n            )\n            annotation_id += 1\n\n    payload = {\n        "images": images,\n        "annotations": annotations,\n        "categories": [\n            {"id": index + 1, "name": class_name, "supercategory": "ppe"}\n            for index, class_name in enumerate(CLASS_NAMES)\n        ],\n    }\n    output_json.parent.mkdir(parents=True, exist_ok=True)\n    output_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n    return {\n        "images": len(images),\n        "instances": len(annotations),\n        "categories": len(CLASS_NAMES),\n    }\n\n\ndef build_oversampled_coco(\n    train_json: Path,\n    output_json: Path,\n    minority_class_ids_zero_based: Iterable[int] = MINORITY_CLASS_IDS_ZERO_BASED,\n    repeat_factor: int = MINORITY_REPEAT_FACTOR,\n) -> dict[str, int]:\n    if repeat_factor < 1:\n        raise ValueError("repeat_factor must be >= 1")\n\n    payload = json.loads(train_json.read_text(encoding="utf-8"))\n    minority_category_ids = {class_id + 1 for class_id in minority_class_ids_zero_based}\n\n    annotations_by_image: dict[int, list[dict]] = {}\n    for annotation in payload["annotations"]:\n        annotations_by_image.setdefault(annotation["image_id"], []).append(annotation)\n\n    output_images = [copy.deepcopy(image) for image in payload["images"]]\n    output_annotations = [copy.deepcopy(annotation) for annotation in payload["annotations"]]\n    next_image_id = max((image["id"] for image in output_images), default=0) + 1\n    next_annotation_id = max((annotation["id"] for annotation in output_annotations), default=0) + 1\n\n    for image in payload["images"]:\n        image_annotations = annotations_by_image.get(image["id"], [])\n        has_minority = any(annotation["category_id"] in minority_category_ids for annotation in image_annotations)\n        if not has_minority:\n            continue\n        for _ in range(repeat_factor - 1):\n            duplicated_image = copy.deepcopy(image)\n            duplicated_image["id"] = next_image_id\n            output_images.append(duplicated_image)\n            for annotation in image_annotations:\n                duplicated_annotation = copy.deepcopy(annotation)\n                duplicated_annotation["id"] = next_annotation_id\n                duplicated_annotation["image_id"] = next_image_id\n                output_annotations.append(duplicated_annotation)\n                next_annotation_id += 1\n            next_image_id += 1\n\n    output_payload = {\n        "images": output_images,\n        "annotations": output_annotations,\n        "categories": payload["categories"],\n    }\n    output_json.parent.mkdir(parents=True, exist_ok=True)\n    output_json.write_text(json.dumps(output_payload, indent=2), encoding="utf-8")\n    return {\n        "images": len(output_images),\n        "instances": len(output_annotations),\n        "categories": len(output_payload["categories"]),\n    }\n\n\ndef build_picodet_config_text(\n    experiment: PicodetExperiment,\n    dataset_dir: Path,\n    output_dir: Path,\n    paddledet_root: Path,\n    epochs: int = 200,\n) -> str:\n    official_config = paddledet_root / "configs" / "picodet" / experiment.official_config_name\n    annotations_dir = output_dir.parent / "dataset" / "annotations"\n    return f"""_BASE_: ["{_path_for_yaml(official_config)}"]\n\nmetric: COCO\nnum_classes: 17\nuse_ema: true\nepoch: {epochs}\nsnapshot_epoch: 10\nsave_dir: {_path_for_yaml(output_dir / experiment.name)}\npretrain_weights: https://paddle-imagenet-models-name.bj.bcebos.com/dygraph/legendary_models/PPLCNet_x2_0_pretrained.pdparams\nweights: {_path_for_yaml(output_dir / experiment.name / "best_model")}\n\nTrainDataset:\n  name: COCODataSet\n  dataset_dir: {_path_for_yaml(dataset_dir)}\n  image_dir: .\n  anno_path: {_path_for_yaml(annotations_dir / experiment.train_json_name)}\n  data_fields: [\'image\', \'gt_bbox\', \'gt_class\', \'is_crowd\']\n\nEvalDataset:\n  name: COCODataSet\n  dataset_dir: {_path_for_yaml(dataset_dir)}\n  image_dir: .\n  anno_path: {_path_for_yaml(annotations_dir / "instances_val.json")}\n  allow_empty: true\n\nTestDataset:\n  name: ImageFolder\n  dataset_dir: {_path_for_yaml(dataset_dir)}\n  anno_path: {_path_for_yaml(annotations_dir / "instances_val.json")}\n\nLearningRate:\n  base_lr: {experiment.base_lr}\n  schedulers:\n  - name: CosineDecay\n    max_epochs: {epochs}\n  - name: LinearWarmup\n    start_factor: 0.1\n    steps: 300\n\nTrainReader:\n  batch_size: {experiment.batch_size}\n\nEvalReader:\n  batch_size: 8\n"""\n\n\ndef write_picodet_configs(\n    config_dir: Path,\n    dataset_dir: Path,\n    output_dir: Path,\n    paddledet_root: Path,\n    epochs: int = 200,\n) -> list[Path]:\n    config_dir.mkdir(parents=True, exist_ok=True)\n    config_paths = []\n    for experiment in picodet_experiments():\n        config_path = config_dir / experiment.config_file_name\n        config_path.write_text(\n            build_picodet_config_text(\n                experiment=experiment,\n                dataset_dir=dataset_dir,\n                output_dir=output_dir,\n                paddledet_root=paddledet_root,\n                epochs=epochs,\n            ),\n            encoding="utf-8",\n        )\n        config_paths.append(config_path)\n    return config_paths\n'


def _candidate_helper_roots():
    roots = [PROJECT_ROOT, PROJECT_ROOT / "SH17"]
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists():
        roots.append(kaggle_working)
    return roots


def _ensure_helper_module():
    for root in _candidate_helper_roots():
        helper_path = root / HELPER_MODULE_RELATIVE_PATH
        if helper_path.exists():
            sys.path.insert(0, str(root))
            return helper_path

    generated_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT
    generated_path = generated_root / HELPER_MODULE_RELATIVE_PATH
    generated_path.parent.mkdir(parents=True, exist_ok=True)
    generated_path.write_text(SELF_CONTAINED_HELPER_SOURCE, encoding="utf-8")
    sys.path.insert(0, str(generated_root))
    return generated_path


HELPER_MODULE_PATH = _ensure_helper_module()
print("Helper module:", HELPER_MODULE_PATH)

from scripts.sh17_picodet_dataset import (
    CLASS_NAMES,
    MINORITY_CLASS_IDS_ZERO_BASED,
    MINORITY_REPEAT_FACTOR,
    PICODET_L_PARAMS_M,
    build_oversampled_coco,
    convert_manifest_to_coco,
    picodet_experiments,
    write_picodet_configs,
)

assert CLASS_NAMES == EXPECTED_CLASS_NAMES, "Class order drifted from the SH17 presentation setup."
print(f"PP-PicoDet-L params: {PICODET_L_PARAMS_M:.2f}M")
print("Minority classes:", [CLASS_NAMES[index] for index in sorted(MINORITY_CLASS_IDS_ZERO_BASED)])


In [ ]:
import random
import subprocess
import textwrap


def run_command(command, cwd=None, check=True):
    print("\n$", " ".join(str(part) for part in command))
    result = subprocess.run(command, cwd=cwd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {command}")
    return result


def ensure_paddledetection():
    if AUTO_SETUP and not PADDLEDET_ROOT.exists():
        PADDLEDET_ROOT.parent.mkdir(parents=True, exist_ok=True)
        run_command([
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/PaddlePaddle/PaddleDetection.git",
            str(PADDLEDET_ROOT),
        ])

    if AUTO_SETUP:
        requirements = PADDLEDET_ROOT / "requirements.txt"
        if requirements.exists():
            run_command([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])

    try:
        import paddle
    except Exception as exc:
        if not AUTO_SETUP:
            raise
        print("Paddle import failed. Trying a default GPU install. If CUDA does not match, install Paddle manually and rerun.")
        run_command([sys.executable, "-m", "pip", "install", "-q", "paddlepaddle-gpu"])
        import paddle

    random.seed(SEED)
    paddle.seed(SEED)
    device = paddle.device.get_device()
    print("Paddle device:", device)
    if USE_GPU and "gpu" not in device.lower():
        raise RuntimeError(
            textwrap.dedent(
                """
                USE_GPU=True but Paddle does not report a GPU device.
                Install a PaddlePaddle GPU wheel matching the machine CUDA version, then rerun this cell.
                """
            ).strip()
        )
    return device


if AUTO_SETUP:
    ensure_paddledetection()
else:
    print("AUTO_SETUP=False: skipping dependency setup.")

In [ ]:
assert TRAIN_MANIFEST.exists(), f"Missing train manifest: {TRAIN_MANIFEST}"
assert VAL_MANIFEST.exists(), f"Missing val manifest: {VAL_MANIFEST}"

DATASET_ARTIFACT_DIR = OUTPUT_ROOT / "dataset"
ANNOTATION_DIR = DATASET_ARTIFACT_DIR / "annotations"
TRAIN_JSON = ANNOTATION_DIR / "instances_train.json"
VAL_JSON = ANNOTATION_DIR / "instances_val.json"
TRAIN_OVERSAMPLED_JSON = ANNOTATION_DIR / "instances_train_oversampled.json"

train_stats = convert_manifest_to_coco(
    manifest_path=TRAIN_MANIFEST,
    dataset_root=DATA_ROOT,
    output_json=TRAIN_JSON,
)
val_stats = convert_manifest_to_coco(
    manifest_path=VAL_MANIFEST,
    dataset_root=DATA_ROOT,
    output_json=VAL_JSON,
)
oversampled_stats = build_oversampled_coco(
    train_json=TRAIN_JSON,
    output_json=TRAIN_OVERSAMPLED_JSON,
    minority_class_ids_zero_based=MINORITY_CLASS_IDS_ZERO_BASED,
    repeat_factor=MINORITY_REPEAT_FACTOR,
)

print("Train COCO:", train_stats)
print("Val COCO:", val_stats)
print("Oversampled train COCO:", oversampled_stats)
print("Annotations:", ANNOTATION_DIR)

In [ ]:
CONFIG_DIR = OUTPUT_ROOT / "configs"
RUNS_DIR = OUTPUT_ROOT / "runs"
CONFIG_PATHS = write_picodet_configs(
    config_dir=CONFIG_DIR,
    dataset_dir=DATA_ROOT,
    output_dir=RUNS_DIR,
    paddledet_root=PADDLEDET_ROOT,
    epochs=EPOCHS,
)

EXPERIMENTS = {experiment.name: experiment for experiment in picodet_experiments()}
CONFIG_BY_EXPERIMENT = {experiment.name: CONFIG_DIR / experiment.config_file_name for experiment in EXPERIMENTS.values()}

for experiment_name in RUN_EXPERIMENT_NAMES:
    experiment = EXPERIMENTS[experiment_name]
    config_path = CONFIG_BY_EXPERIMENT[experiment_name]
    print(f"{experiment_name}: input={experiment.input_size}, batch={experiment.batch_size}, lr={experiment.base_lr}, config={config_path}")

In [ ]:
def _checkpoint_exists(base_path):
    return base_path.exists() or base_path.with_suffix(".pdparams").exists()


def _numeric_checkpoint_bases(run_dir):
    checkpoints = []
    for checkpoint_path in run_dir.glob("*.pdparams"):
        if checkpoint_path.stem.isdigit():
            checkpoints.append((int(checkpoint_path.stem), checkpoint_path.with_suffix("")))
    return [base_path for _, base_path in sorted(checkpoints, reverse=True)]


def checkpoint_base_path(run_dir):
    candidates = [
        run_dir / "best_model",
        run_dir / "model_final",
        *_numeric_checkpoint_bases(run_dir),
    ]
    for candidate in candidates:
        if _checkpoint_exists(candidate):
            return candidate
    return run_dir / "best_model"


def last_checkpoint_base_path(run_dir):
    candidates = [
        run_dir / "model_final",
        *_numeric_checkpoint_bases(run_dir),
        run_dir / "best_model",
    ]
    for candidate in candidates:
        if _checkpoint_exists(candidate):
            return candidate
    return None


def train_experiment(experiment_name):
    config_path = CONFIG_BY_EXPERIMENT[experiment_name]
    run_dir = RUNS_DIR / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)

    command = [
        sys.executable,
        "tools/train.py",
        "-c",
        str(config_path),
        "--eval",
    ]
    last_checkpoint = last_checkpoint_base_path(run_dir)
    if last_checkpoint is not None:
        command.extend(["-r", str(last_checkpoint)])
    return run_command(command, cwd=PADDLEDET_ROOT)


def evaluate_experiment(experiment_name):
    config_path = CONFIG_BY_EXPERIMENT[experiment_name]
    run_dir = RUNS_DIR / experiment_name
    eval_dir = OUTPUT_ROOT / "eval" / experiment_name
    eval_dir.mkdir(parents=True, exist_ok=True)
    weights = checkpoint_base_path(run_dir)
    command = [
        sys.executable,
        "tools/eval.py",
        "-c",
        str(config_path),
        "-o",
        f"weights={weights}",
        "--classwise",
        "--output_eval",
        str(eval_dir),
    ]
    return run_command(command, cwd=PADDLEDET_ROOT)


if RUN_TRAINING:
    for experiment_name in RUN_EXPERIMENT_NAMES:
        train_experiment(experiment_name)
        evaluate_experiment(experiment_name)
else:
    print("RUN_TRAINING=False. Review configs and set RUN_TRAINING=True on the training machine.")


In [ ]:
import csv
import re


def collect_result_rows():
    rows = []
    for experiment_name in RUN_EXPERIMENT_NAMES:
        experiment = EXPERIMENTS[experiment_name]
        eval_dir = OUTPUT_ROOT / "eval" / experiment_name
        row = {
            "experiment": experiment_name,
            "input_size": experiment.input_size,
            "params_m": PICODET_L_PARAMS_M,
            "batch_size": experiment.batch_size,
            "base_lr": experiment.base_lr,
            "train_json": experiment.train_json_name,
            "bbox_json": str(eval_dir / "bbox.json"),
        }
        rows.append(row)
    return rows


rows = collect_result_rows()
summary_csv = OUTPUT_ROOT / "picodet_l_experiment_matrix.csv"
with summary_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Wrote experiment matrix:", summary_csv)
for row in rows:
    print(row)

## Presentation note

Use this comparison in the report:

`PP-PicoDet-L` is a non-YOLO and non-EfficientDet detector with about `5.80M` parameters. The official PaddleDetection model zoo provides 320, 416, and 640 input variants, so the SH17 ablation can test whether increasing input resolution and oversampling rare PPE classes improves the same failure modes observed in YOLOv9s: small objects and class imbalance.